In [4]:
import numpy as np
from scipy.ndimage import rotate
from plotly.subplots import make_subplots
import plotly.graph_objects as go

N = 128

img = np.zeros((N, N))

# barre horizontale
img[20:35, 30:98] = 1

# barre verticale
img[35:100, 54:74] = 1

img = np.rot90(img, 2)

angles = [0, 45, 90, 135]

backprojections = []

for theta in angles:

    # Projection
    rotated = rotate(
        img,
        theta,
        reshape=False,
        order=1
    )

    projection = rotated.sum(axis=0)
    bp = np.tile(projection, (N, 1))

    bp = rotate(
        bp,
        -theta,
        reshape=False,
        order=1
    )

    backprojections.append(bp)
reconstruction = np.sum(backprojections, axis=0)

reconstruction /= reconstruction.max()

fig = make_subplots(
    rows=2,
    cols=3,
    vertical_spacing=0.15,  # espace entre les deux lignes
    subplot_titles=[
        r"$\large{\text{Objet}}$",
        r"$\large{0^\circ}$",
        r"$\large{45^\circ}$",
        r"$\large{90^\circ}$",
        r"$\large{135^\circ}$",
        r"$\large{\text{Somme}}$"
    ]
)

# Objet original
fig.add_trace(
    go.Heatmap(
        z=img,
        colorscale="Greys",
        showscale=False
    ),
    row=1,
    col=1
)

# Back projections
positions = [
    (1, 2),
    (1, 3),
    (2, 1),
    (2, 2)
]

for bp, pos in zip(backprojections, positions):

    fig.add_trace(
        go.Heatmap(
            z=bp,
            colorscale="gray",
            showscale=False
        ),
        row=pos[0],
        col=pos[1]
    )

# Somme
fig.add_trace(
    go.Heatmap(
        z=reconstruction,
        colorscale="gray",
        showscale=False
    ),
    row=2,
    col=3
)

for i in range(1, 3):
    for j in range(1, 4):
        fig.update_xaxes(showticklabels=False, row=i, col=j)
        fig.update_yaxes(showticklabels=False, row=i, col=j)

fig.update_layout(
    template="plotly_white",

)

fig.update_layout(
    template="plotly_white",
)

for ann in fig.layout.annotations:
    ann.y += 0.03

fig.write_image("images/5 fig1 lr.webp", scale=1.3, height=600, width=1038)
fig.write_image("images/5 fig1 hr.png", scale=10, height=600, width=1038)

In [12]:
import numpy as np
from scipy.ndimage import rotate
from plotly.subplots import make_subplots
import plotly.graph_objects as go

N = 128
img = np.zeros((N, N))
img[20:35, 30:98] = 1  # barre horizontale
img[35:100, 54:74] = 1  # barre verticale


angles = np.arange(0, 180, 5)  # 36 projections pour bien remplir l'espace
F_domain = np.zeros((N, N), dtype=complex)
center = N // 2

# Vecteur de coordonnées pour la tranche 1D
r = np.arange(-center, center)

for theta in angles:
    # 1. Projection
    rotated = rotate(img, theta, reshape=False, order=1)
    projection = rotated.sum(axis=0)

    # 2. Transformée de Fourier 1D de la projection
    # fftshift recentre les basses fréquences au milieu
    proj_fft = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(projection)))

    # 3. Placement radial dans l'espace de Fourier 2D
    theta_rad = np.deg2rad(theta)

    # Conversion polaire -> cartésien (Arrondi au pixel le plus proche pour l'illustration)
    x = np.round(center + r * np.cos(theta_rad)).astype(int)
    y = np.round(center - r * np.sin(theta_rad)).astype(int)  # - car l'axe y descend

    # On ne garde que les indices qui restent dans l'image
    valid = (x >= 0) & (x < N) & (y >= 0) & (y < N)

    # On "dessine" la ligne de fréquences dans notre matrice 2D
    F_domain[y[valid], x[valid]] = proj_fft[valid]

reconstruction = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(F_domain))).real
reconstruction = np.maximum(reconstruction, 0)  # Supprime les petits artefacts négatifs
reconstruction /= reconstruction.max()

F_original = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(img)))

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        r"$\large{\text{Objet Original}}$",
        r"$\large{\text{Spectre Réel}}$",
        r"$\large{\text{Espace de Fourier en étoile}}$",
        r"$\large{\text{Reconstruction Spatiale}}$"
    ]
)

# 1. Objet Original
fig.add_trace(go.Heatmap(z=np.rot90(img, 2), colorscale="Greys", showscale=False), row=1, col=1)

# 2. Spectre Original (on utilise np.log pour voir les contrastes des fréquences)
fig.add_trace(go.Heatmap(z=np.log(1 + np.abs(F_original)), colorscale="Viridis", showscale=False), row=1, col=2)

# 3. Espace de Fourier Reconstruit (Les fameuses "tranches")
fig.add_trace(go.Heatmap(z=np.log(1 + np.abs(F_domain)), colorscale="Viridis", showscale=False), row=2, col=1)

# 4. Image Reconstruite
fig.add_trace(go.Heatmap(z=reconstruction, colorscale="RdYlGn", showscale=False), row=2, col=2)

# Nettoyage des axes
for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(showticklabels=False, row=i, col=j)
        fig.update_yaxes(showticklabels=False, row=i, col=j)

fig.update_layout(
    template="plotly_white",
)

fig.write_image("images/5 fig2 lr.webp", scale=1.3, width=1170 / 2, height=1213 / 2)
fig.write_image("images/5 fig2 hr.png", width=1170 / 2, height=1213 / 2)